# Delta Demo — Day 2: Insert More Rows
**Prerequisite:** Delta_Demo_Day1_Load must already have been run — this notebook assumes the `employees` table already exists with Day 1's 3 rows.

Adds 2 new employees, then verifies the effect on disk: a new commit, a new Parquet file, and Day 1's original file completely untouched.

## Day 2 — Insert New Rows (eno 4, 5)
Now we add two more employees. Watch what happens to the FILES on disk when we do this — an INSERT should add a brand new file without touching anything from Day 1.

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/employees` VALUES
(4, 'Srik', 32000),
(5, 'Kanth', 14999);

## Day 2 — Raw Contents Deep Dive (New Commit + Two Files Now on Disk)
Same technique as Day 1 — read the new commit's JSON directly, then read every Parquet file on disk. You should now see version 1's commit (mode = Append), and TWO physical files: Day 1's original file completely unchanged, plus a brand new file holding only the two new rows.

In [0]:
%sh
# Version 1 is the commit this INSERT just created — read it the same NDJSON-aware way.
cat /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/00000000000000000001.json \
  | while IFS= read -r line; do echo "$line" | python3 -m json.tool; echo "---"; done

In [0]:
%python
# Read every physical Parquet file again. Day 1's file should be byte-for-byte
# identical to before — proof that INSERT never rewrites existing files, it only
# ever adds new ones.
import pandas as pd, glob
for f in sorted(glob.glob("/Volumes/workspace/delta_demo/demo_files/employees/*.parquet")):
    print(f"\n=== {f.split('/')[-1]} ===")
    print(pd.read_parquet(f))

## Verify Day 2 Results
Query through Delta again — this always reflects the CURRENT logical state, regardless of how many physical files are involved behind the scenes.

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees` ORDER BY eno;

## Describe History — After Day 2
You should now see TWO rows in the history: the original WRITE (version 0) and this new Append (version 1).

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees`;

## Browse `_delta_log` — After Day 2 (Quick File Count Check)
A fast way to just eyeball how many commit files exist, without reading their full contents — useful as a running sanity check as we move through more operations.

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/*.json

## Day 2 Summary — List All Objects
Full listing after Day 2 — you should now see 2 JSON commits and 2 Parquet files.

In [0]:
%sh
echo "=== _delta_log/ ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees/_delta_log/
echo ""
echo "=== data files (Parquet + deletion vectors) ==="
ls -la /Volumes/workspace/delta_demo/demo_files/employees/ | grep -v _delta_log